# 09 — Internações Potencialmente Desnecessárias (RQ9)

**Pergunta:** Como identificar e quantificar internações desnecessárias?

Notebook 05 introduziu um score composto. Aqui aprofundamos:
1. Refinamento do score e definição dos indicadores
2. Quem faz mais / menos (ranking hospitalar)
3. Tipo de instituição (Santa Casa, pública, privada, universitária, AME)
4. Por quê — incentivo financeiro, falta de infraestrutura, padrão clínico
5. Impacto real — financeiro, leitos-dia, custo de oportunidade

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from shared import (
    load_kidney, load_hospital_tags, load_cnes_enriched, load_cnes_names,
    hospital_name, classify_legal_nature, setup_plot_style, save_plot,
    save_metrics, COLORS, PROC_NAMES, RAW_SIA_DIR,
)

setup_plot_style()

kidney = load_kidney()
tags = load_hospital_tags()
cnes = load_cnes_enriched()
names_df = load_cnes_names()

kidney = kidney.merge(tags[["CNES", "facility_type", "admission_profile", "case_mix_profile", "comparability_group"]],
                      on="CNES", how="left")

# Legal nature from CNES enriched (NAT_JUR) — SIH NATUREZA is always "00"
if "NAT_JUR" in cnes.columns:
    nat_map = cnes.drop_duplicates("CNES").set_index("CNES")["NAT_JUR"].astype(str).str.strip()
    kidney["legal_nature"] = kidney["CNES"].map(nat_map).apply(classify_legal_nature)
else:
    kidney["legal_nature"] = "unknown"

print(f"Dataset: {len(kidney):,} admissions, {kidney['CNES'].nunique()} hospitals")
print(f"Legal nature distribution:\n{kidney['legal_nature'].value_counts().to_string()}")

Dataset: 206,500 admissions, 510 hospitals
Legal nature distribution:
legal_nature
public              100724
assoc_privada        75091
fundacao_privada     30215
private                292
unknown                178


## 1. Definição dos Indicadores de Desnecessidade

O score v1 (notebook 05) tinha um bug: D0 e baixo custo contavam para TODAS as internações,
punindo hospitais que fazem cirurgia e dão alta no mesmo dia (eficiência, não desperdício).

**Score v2 — correção:** D0 e baixo custo só contam se o procedimento NÃO foi terapêutico.

| Flag | Critério | Justificativa |
|---|---|---|
| `sem_tratamento` | proc_category ∈ {DIAGNOSTIC, OBSERVATION} | Não houve intervenção terapêutica |
| `sia_available` | PROC_REA existe no SIA | Mesmo procedimento é feito ambulatorialmente |
| `d0_sem_terapia` | DIAS_PERM == 0 AND proc_category ∉ {SURGICAL, INTERVENTIONAL, SURGICAL_MGMT} | D0 sem cirurgia/intervenção |
| `baixo_custo_sem_terapia` | VAL_TOT < P25 AND proc_category ∉ {SURGICAL, INTERVENTIONAL, SURGICAL_MGMT} | Custo baixo em caso não-terapêutico |

Um D0 após ureteroscopia = eficiência (score += 0). Um D0 após urografia = candidato ambulatorial (score += 1).

In [2]:
# --- Build SIA procedure set ---
sia_files = sorted(RAW_SIA_DIR.glob("PASP*.parquet"))
if sia_files:
    sia_procs = set()
    for f in sia_files:
        try:
            df = pd.read_parquet(f, columns=["PA_PROC_ID"])
            sia_procs.update(df["PA_PROC_ID"].astype(str).str.strip().unique())
        except Exception:
            pass
    kidney_procs = set(kidney["PROC_REA"].unique())
    sia_overlap = kidney_procs & sia_procs
    print(f"SIA procedures loaded: {len(sia_procs):,}")
    print(f"Kidney procedures in SIA: {len(sia_overlap)} of {len(kidney_procs)}")
else:
    sia_overlap = set()
    print("No SIA files found — flag_sia_available will be 0 for all")

# --- Therapeutic vs non-therapeutic ---
THERAPEUTIC = {"SURGICAL", "INTERVENTIONAL", "SURGICAL_MGMT"}
kidney["is_therapeutic"] = kidney["proc_category"].isin(THERAPEUTIC).astype(int)

print(f"\nProcedure split:")
print(f"  Terapêutico (cirúrgico/intervencionista/man.cirúrgico): "
      f"{kidney['is_therapeutic'].sum():,} ({kidney['is_therapeutic'].mean()*100:.1f}%)")
print(f"  Não-terapêutico (diagnóstico/observação/clínico/outro): "
      f"{(~kidney['is_therapeutic'].astype(bool)).sum():,} ({(~kidney['is_therapeutic'].astype(bool)).mean()*100:.1f}%)")

# --- Build flags (v2 — corrigido) ---
# Flags de "motivo fraco para internar"
kidney["flag_no_treatment"] = kidney["proc_category"].isin({"DIAGNOSTIC", "OBSERVATION"}).astype(int)
kidney["flag_sia"] = kidney["PROC_REA"].isin(sia_overlap).astype(int)

# Flags condicionais — só contam se NÃO houve terapia
kidney["flag_d0_no_therapy"] = (
    (kidney["DIAS_PERM"] == 0) & (~kidney["is_therapeutic"].astype(bool))
).astype(int)
kidney["flag_lowcost_no_therapy"] = (
    (kidney["VAL_TOT"] < kidney["VAL_TOT"].quantile(0.25)) & 
    (~kidney["is_therapeutic"].astype(bool))
).astype(int)

kidney["unnecessary_score"] = (
    kidney["flag_no_treatment"] +
    kidney["flag_sia"] +
    kidney["flag_d0_no_therapy"] +
    kidney["flag_lowcost_no_therapy"]
)

# --- Compare v1 vs v2 ---
# v1 score (old, for comparison)
kidney["v1_d0"] = (kidney["DIAS_PERM"] == 0).astype(int)
kidney["v1_lowcost"] = (kidney["VAL_TOT"] < kidney["VAL_TOT"].quantile(0.25)).astype(int)
kidney["v1_score"] = (
    (kidney["proc_category"] == "DIAGNOSTIC").astype(int) +
    (kidney["proc_category"] == "OBSERVATION").astype(int) +
    kidney["v1_d0"] +
    kidney["flag_sia"] +
    kidney["v1_lowcost"]
)

print(f"\n=== SCORE v2 (corrigido) vs v1 (antigo) ===")
print(f"{'Score':>5s} {'v2 (n)':>10s} {'v2 (%)':>7s} {'v1 (n)':>10s} {'v1 (%)':>7s}")
print("-" * 45)
for score in range(5):
    v2_n = (kidney["unnecessary_score"] == score).sum()
    v1_n = (kidney["v1_score"] == score).sum()
    print(f"  {score:>3d} {v2_n:>10,} {v2_n/len(kidney)*100:6.1f}% {v1_n:>10,} {v1_n/len(kidney)*100:6.1f}%")

v2_high = (kidney["unnecessary_score"] >= 3).sum()
v1_high = (kidney["v1_score"] >= 3).sum()
print(f"\n  Alta suspeita (≥3):")
print(f"    v2: {v2_high:,} ({v2_high/len(kidney)*100:.1f}%)")
print(f"    v1: {v1_high:,} ({v1_high/len(kidney)*100:.1f}%)")

# Surgical D0 that v1 wrongly flagged
surgical_d0 = kidney[(kidney["is_therapeutic"] == 1) & (kidney["DIAS_PERM"] == 0)]
print(f"\n  D0 cirúrgico/terapêutico (v1 punia, v2 não): {len(surgical_d0):,}")
print(f"    Eram score ≥3 no v1: {(surgical_d0['v1_score'] >= 3).sum():,}")
print(f"    São score ≥3 no v2:  {(surgical_d0['unnecessary_score'] >= 3).sum():,}")

print(f"\n=== SCORE v2 DISTRIBUTION ===")
for score in sorted(kidney["unnecessary_score"].unique()):
    sub = kidney[kidney["unnecessary_score"] == score]
    print(f"  Score {score}: {len(sub):>7,} ({len(sub)/len(kidney)*100:5.1f}%) "
          f"| R${sub['VAL_TOT'].sum():>12,.0f} | LOS={sub['DIAS_PERM'].mean():.1f}d "
          f"| mort={sub['MORTE'].mean()*100:.2f}% "
          f"| terap={sub['is_therapeutic'].mean()*100:.0f}%")

SIA procedures loaded: 2,668
Kidney procedures in SIA: 15 of 193

Procedure split:
  Terapêutico (cirúrgico/intervencionista/man.cirúrgico): 129,391 (62.7%)
  Não-terapêutico (diagnóstico/observação/clínico/outro): 77,109 (37.3%)



=== SCORE v2 (corrigido) vs v1 (antigo) ===
Score     v2 (n)  v2 (%)     v1 (n)  v1 (%)
---------------------------------------------
    0    109,893   53.2%     97,984   47.4%
    1     62,165   30.1%     56,996   27.6%
    2     27,828   13.5%     42,284   20.5%
    3      6,614    3.2%      9,236    4.5%
    4          0    0.0%          0    0.0%

  Alta suspeita (≥3):
    v2: 6,614 (3.2%)
    v1: 9,236 (4.5%)

  D0 cirúrgico/terapêutico (v1 punia, v2 não): 14,864
    Eram score ≥3 no v1: 2,622
    São score ≥3 no v2:  0

=== SCORE v2 DISTRIBUTION ===
  Score 0: 109,893 ( 53.2%) | R$ 135,138,061 | LOS=2.3d | mort=0.25% | terap=80%
  Score 1:  62,165 ( 30.1%) | R$  45,706,223 | LOS=3.0d | mort=0.63% | terap=67%
  Score 2:  27,828 ( 13.5%) | R$   6,041,681 | LOS=2.2d | mort=0.14% | terap=0%
  Score 3:   6,614 (  3.2%) | R$     937,190 | LOS=0.0d | mort=0.09% | terap=0%


## 2. Quem Faz Mais? Quem Faz Menos? — Ranking Hospitalar

In [3]:
MIN_N = 30

hosp_unnec = kidney.groupby("CNES").agg(
    n=("CNES", "count"),
    mean_score=("unnecessary_score", "mean"),
    high_sus_n=("unnecessary_score", lambda x: (x >= 3).sum()),
    high_sus_pct=("unnecessary_score", lambda x: (x >= 3).mean()),
    pct_no_treatment=("flag_no_treatment", "mean"),
    pct_d0_no_therapy=("flag_d0_no_therapy", "mean"),
    pct_sia=("flag_sia", "mean"),
    pct_lowcost_no_therapy=("flag_lowcost_no_therapy", "mean"),
    pct_therapeutic=("is_therapeutic", "mean"),
    total_cost=("VAL_TOT", "sum"),
    mean_cost=("VAL_TOT", "mean"),
    mean_los=("DIAS_PERM", "mean"),
    mortality=("MORTE", "mean"),
    pct_emergency=("is_emergency", "mean"),
).reset_index()
hosp_unnec = hosp_unnec[hosp_unnec["n"] >= MIN_N].copy()
hosp_unnec["name"] = hosp_unnec["CNES"].apply(lambda c: hospital_name(c, names_df))

# Merge legal nature and facility type
legal_map = kidney.drop_duplicates("CNES").set_index("CNES")["legal_nature"]
factype_map = kidney.drop_duplicates("CNES").set_index("CNES")["facility_type"]
hosp_unnec["legal_nature"] = hosp_unnec["CNES"].map(legal_map).fillna("unknown")
hosp_unnec["facility_type"] = hosp_unnec["CNES"].map(factype_map).fillna("unknown")

hosp_unnec = hosp_unnec.sort_values("high_sus_pct", ascending=False)

print(f"Hospitals analyzed (≥{MIN_N} admissions): {len(hosp_unnec)}")
print(f"\n{'='*140}")
print("TOP 20 — MAIOR % DE INTERNAÇÕES SUSPEITAS (score ≥ 3)")
print(f"{'='*140}")
hdr = (f"{'Hospital':40s} {'Nat.':>14s} {'N':>5s} "
       f"{'Susp%':>6s} {'SemTer%':>7s} {'D0nt%':>6s} {'SIA%':>5s} "
       f"{'Terap%':>6s} {'LOS':>5s} {'Mort%':>6s}")
print(hdr)
print("-" * 120)
for _, r in hosp_unnec.head(20).iterrows():
    print(f"{r['name'][:40]:40s} {str(r['legal_nature'])[:14]:>14s} "
          f"{r['n']:5.0f} {r['high_sus_pct']*100:5.1f}% {r['pct_no_treatment']*100:6.1f}% "
          f"{r['pct_d0_no_therapy']*100:5.1f}% {r['pct_sia']*100:4.0f}% "
          f"{r['pct_therapeutic']*100:5.1f}% {r['mean_los']:5.1f} {r['mortality']*100:5.2f}%")

print(f"\n{'='*120}")
print("TOP 20 — MENOR % DE INTERNAÇÕES SUSPEITAS")
print(f"{'='*120}")
print(hdr)
print("-" * 120)
for _, r in hosp_unnec.tail(20).sort_values("high_sus_pct").iterrows():
    print(f"{r['name'][:40]:40s} {str(r['legal_nature'])[:14]:>14s} "
          f"{r['n']:5.0f} {r['high_sus_pct']*100:5.1f}% {r['pct_no_treatment']*100:6.1f}% "
          f"{r['pct_d0_no_therapy']*100:5.1f}% {r['pct_sia']*100:4.0f}% "
          f"{r['pct_therapeutic']*100:5.1f}% {r['mean_los']:5.1f} {r['mortality']*100:5.2f}%")

Hospitals analyzed (≥30 admissions): 283

TOP 20 — MAIOR % DE INTERNAÇÕES SUSPEITAS (score ≥ 3)
Hospital                                           Nat.     N  Susp% SemTer%  D0nt%  SIA% Terap%   LOS  Mort%
------------------------------------------------------------------------------------------------------------------------
HOSPITAL UNIVERSITARIO DA UFSCAR                private    59  39.0%   69.5%  61.0%    0%   8.5%   0.9  0.00%
SANTA CASA DE MONTE ALTO                  assoc_privada   393  28.5%   36.1%  32.8%   28%  52.4%   0.9  0.00%
HOSPITAL SAO LUIZ                        fundacao_priva   137  25.5%   75.9%  27.7%   20%  21.9%   1.4  0.00%
UNIDADE DE RETAGUARDA DE URGENCIA E DIAG         public    48  25.0%  100.0%  25.0%    0%   0.0%   1.7  0.00%
SANTA CASA DE VINHEDO                     assoc_privada  1101  22.5%   28.5%  47.1%    7%  19.4%   1.2  0.36%
HOSPITAL UNIVERSITARIO SAO FRANCISCO NA   assoc_privada  1975  22.3%   30.0%  22.4%   21%  68.6%   1.6  0.56%
HOSPITAL GERA

## 3. Tipo de Instituição — Quem Faz Mais Internações Desnecessárias?

In [4]:
# --- By legal nature ---
print("=== POR NATUREZA JURÍDICA ===")
legal_stats = kidney.groupby("legal_nature").agg(
    n=("CNES", "count"),
    hospitals=("CNES", "nunique"),
    mean_score=("unnecessary_score", "mean"),
    high_sus_pct=("unnecessary_score", lambda x: (x >= 3).mean()),
    pct_no_treatment=("flag_no_treatment", "mean"),
    pct_therapeutic=("is_therapeutic", "mean"),
    mean_los=("DIAS_PERM", "mean"),
    mortality=("MORTE", "mean"),
    mean_cost=("VAL_TOT", "mean"),
).sort_values("high_sus_pct", ascending=False)

print(f"{'Natureza':20s} {'N':>8s} {'Hosp':>5s} {'Score':>6s} {'Susp%':>6s} "
      f"{'SemTer%':>7s} {'Terap%':>7s} {'LOS':>5s} {'Mort%':>6s} {'Custo':>8s}")
print("-" * 95)
for nat, r in legal_stats.iterrows():
    print(f"{nat:20s} {r['n']:>8,.0f} {r['hospitals']:5.0f} {r['mean_score']:5.2f} "
          f"{r['high_sus_pct']*100:5.1f}% {r['pct_no_treatment']*100:6.1f}% "
          f"{r['pct_therapeutic']*100:6.1f}% {r['mean_los']:5.1f} {r['mortality']*100:5.2f}% "
          f"R${r['mean_cost']:>6,.0f}")

# --- By facility type ---
print(f"\n=== POR TIPO DE UNIDADE ===")
fac_stats = kidney.groupby("facility_type").agg(
    n=("CNES", "count"),
    hospitals=("CNES", "nunique"),
    mean_score=("unnecessary_score", "mean"),
    high_sus_pct=("unnecessary_score", lambda x: (x >= 3).mean()),
    pct_no_treatment=("flag_no_treatment", "mean"),
    pct_therapeutic=("is_therapeutic", "mean"),
    mean_los=("DIAS_PERM", "mean"),
    mortality=("MORTE", "mean"),
    mean_cost=("VAL_TOT", "mean"),
).sort_values("high_sus_pct", ascending=False)

print(f"{'Tipo':20s} {'N':>8s} {'Hosp':>5s} {'Score':>6s} {'Susp%':>6s} "
      f"{'SemTer%':>7s} {'Terap%':>7s} {'LOS':>5s} {'Mort%':>6s} {'Custo':>8s}")
print("-" * 95)
for fac, r in fac_stats.iterrows():
    print(f"{str(fac)[:20]:20s} {r['n']:>8,.0f} {r['hospitals']:5.0f} {r['mean_score']:5.2f} "
          f"{r['high_sus_pct']*100:5.1f}% {r['pct_no_treatment']*100:6.1f}% "
          f"{r['pct_therapeutic']*100:6.1f}% {r['mean_los']:5.1f} {r['mortality']*100:5.2f}% "
          f"R${r['mean_cost']:>6,.0f}")

# Statistical test: legal nature effect on unnecessary score
groups = [g["unnecessary_score"].values for _, g in kidney.groupby("legal_nature") if len(g) >= 100]
if len(groups) >= 2:
    h_stat, p_val = stats.kruskal(*groups)
    print(f"\nKruskal-Wallis (legal nature → unnecessary score): H={h_stat:.1f}, p={p_val:.2e}")

=== POR NATUREZA JURÍDICA ===


Natureza                    N  Hosp  Score  Susp% SemTer%  Terap%   LOS  Mort%    Custo
-----------------------------------------------------------------------------------------------
private                   292     5  0.43   8.6%   15.1%   79.8%   1.5  0.00% R$   652
assoc_privada          75,091   257  0.74   4.0%   25.0%   60.7%   2.4  0.45% R$   896
public                100,724   211  0.65   2.8%   26.6%   62.2%   2.5  0.29% R$   881
fundacao_privada       30,215    23  0.54   2.4%   15.1%   69.3%   2.4  0.27% R$ 1,045
unknown                   178    14  1.44   1.1%   62.9%   31.5%   3.7  0.00% R$   482

=== POR TIPO DE UNIDADE ===
Tipo                        N  Hosp  Score  Susp% SemTer%  Terap%   LOS  Mort%    Custo
-----------------------------------------------------------------------------------------------
unidade_mista              39     7  2.08  12.8%   94.9%    0.0%   1.9  0.00% R$   175
hospital_geral        194,988   424  0.69   3.4%   25.1%   61.3%   2.5  0.36% R$ 

In [5]:
# --- Visualization: institution type comparison ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. By legal nature
legal_order = legal_stats.sort_values("high_sus_pct", ascending=True).index
colors_legal = [COLORS["danger"] if v > legal_stats["high_sus_pct"].median() else COLORS["primary"] 
                for v in legal_stats.loc[legal_order, "high_sus_pct"]]
axes[0].barh(range(len(legal_order)), legal_stats.loc[legal_order, "high_sus_pct"] * 100, color=colors_legal)
axes[0].set_yticks(range(len(legal_order)))
axes[0].set_yticklabels(legal_order, fontsize=9)
axes[0].set_xlabel("% Internações Suspeitas (score ≥ 3)")
axes[0].set_title("Por Natureza Jurídica")

# 2. By facility type
fac_order = fac_stats.sort_values("high_sus_pct", ascending=True).index
colors_fac = [COLORS["danger"] if v > fac_stats["high_sus_pct"].median() else COLORS["primary"] 
              for v in fac_stats.loc[fac_order, "high_sus_pct"]]
axes[1].barh(range(len(fac_order)), fac_stats.loc[fac_order, "high_sus_pct"] * 100, color=colors_fac)
axes[1].set_yticks(range(len(fac_order)))
axes[1].set_yticklabels(fac_order, fontsize=9)
axes[1].set_xlabel("% Internações Suspeitas (score ≥ 3)")
axes[1].set_title("Por Tipo de Unidade")

# 3. Score distribution by legal nature (top 4)
top_nats = legal_stats.head(4).index
for nat in top_nats:
    sub = kidney[kidney["legal_nature"] == nat]["unnecessary_score"]
    counts = sub.value_counts(normalize=True).sort_index()
    axes[2].plot(counts.index, counts.values * 100, marker="o", label=nat, linewidth=2)
axes[2].set_xlabel("Score de Desnecessidade")
axes[2].set_ylabel("% das Internações")
axes[2].set_title("Distribuição do Score")
axes[2].legend(fontsize=8)

fig.suptitle("Internações Desnecessárias por Tipo de Instituição", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "institution_type_unnecessary", prefix="09")

  Saved: 13_institution_type_unnecessary.png


## 4. Por Quê? — Drivers de Internações Desnecessárias

In [6]:
# --- H9.1: Financial incentive — SIH/SIA premium drives unnecessary admissions? ---
# Load admission premium from nb05 metrics if available
try:
    from shared import load_metrics
    fin_metrics = load_metrics("05_financial_analysis")
    print(f"Max admission premium (SIH/SIA): {fin_metrics.get('max_admission_premium', 'N/A')}x")
except Exception:
    pass

# For each hospital, compute the % of admissions that use SIA-available procedures
# and correlate with unnecessary score
if sia_overlap:
    rho, p = stats.spearmanr(hosp_unnec["pct_sia"], hosp_unnec["high_sus_pct"])
    print(f"\nH9.1: % SIA-available procs vs % unnecessary (Spearman)")
    print(f"  ρ = {rho:.3f}, p = {p:.2e}")
    print(f"  → {'Significant' if p < 0.05 else 'Not significant'}: "
          f"hospitals using more SIA-available procedures {'do' if rho > 0 else 'do NOT'} "
          f"have higher unnecessary rates")

# --- H9.2: Infrastructure gap — diagnostic admissions where no AME exists? ---
# Proxy: hospitals with high diagnostic % AND high emergency %
diag_emerg = hosp_unnec[(hosp_unnec["pct_no_treatment"] > 0.3) & (hosp_unnec["pct_emergency"] > 0.7)]
other = hosp_unnec[~hosp_unnec.index.isin(diag_emerg.index)]
print(f"\nH9.2: Hospitals with >30% diagnostic AND >70% emergency: {len(diag_emerg)}")
print(f"  Their mean unnecessary rate: {diag_emerg['high_sus_pct'].mean()*100:.1f}%")
print(f"  Others:                      {other['high_sus_pct'].mean()*100:.1f}%")
if len(diag_emerg) >= 5 and len(other) >= 5:
    u_stat, u_p = stats.mannwhitneyu(diag_emerg["high_sus_pct"], other["high_sus_pct"], alternative="greater")
    print(f"  Mann-Whitney U: U={u_stat:.0f}, p={u_p:.2e}")

# --- Decompose: which flag drives the most? ---
print(f"\n=== DECOMPOSIÇÃO: PREVALÊNCIA DE CADA FLAG (v2) ===")
flags = ["flag_no_treatment", "flag_sia", "flag_d0_no_therapy", "flag_lowcost_no_therapy"]
flag_labels = ["Sem tratamento (diag/obs)", "Disponível SIA", "D0 sem terapia", "Baixo custo sem terapia"]
for flag, label in zip(flags, flag_labels):
    pct = kidney[flag].mean() * 100
    n = kidney[flag].sum()
    # Of those with score ≥ 3, what % have this flag?
    high_sub = kidney[kidney["unnecessary_score"] >= 3]
    pct_in_high = high_sub[flag].mean() * 100 if len(high_sub) > 0 else 0
    print(f"  {label:20s}: {pct:5.1f}% geral ({n:>7,}) | {pct_in_high:5.1f}% entre suspeitas")

Max admission premium (SIH/SIA): 1752.0x

H9.1: % SIA-available procs vs % unnecessary (Spearman)
  ρ = 0.032, p = 5.88e-01
  → Not significant: hospitals using more SIA-available procedures do have higher unnecessary rates

H9.2: Hospitals with >30% diagnostic AND >70% emergency: 139
  Their mean unnecessary rate: 4.5%
  Others:                      3.5%
  Mann-Whitney U: U=11291, p=3.07e-02

=== DECOMPOSIÇÃO: PREVALÊNCIA DE CADA FLAG (v2) ===
  Sem tratamento (diag/obs):  24.4% geral ( 50,305) |  99.7% entre suspeitas
  Disponível SIA      :  20.2% geral ( 41,643) |   0.3% entre suspeitas
  D0 sem terapia      :   5.4% geral ( 11,159) | 100.0% entre suspeitas
  Baixo custo sem terapia:  16.7% geral ( 34,556) | 100.0% entre suspeitas


In [7]:
# --- H9.3: Financial vs bed-day impact ---
high_sus = kidney[kidney["unnecessary_score"] >= 3]
mod_sus = kidney[kidney["unnecessary_score"] == 2]

total_cost = kidney["VAL_TOT"].sum()
total_beddays = kidney["DIAS_PERM"].sum()
total_adm = len(kidney)

print("=== H9.3: IMPACTO — CUSTO vs LEITOS-DIA ===")
for label, sub in [("Alta suspeita (≥3)", high_sus), ("Moderada (=2)", mod_sus)]:
    n = len(sub)
    cost = sub["VAL_TOT"].sum()
    beddays = sub["DIAS_PERM"].sum()
    print(f"\n  {label}:")
    print(f"    Internações:  {n:>8,} ({n/total_adm*100:.1f}% do total)")
    print(f"    Custo:        R${cost:>12,.0f} ({cost/total_cost*100:.1f}% do total)")
    print(f"    Leitos-dia:   {beddays:>8,.0f} ({beddays/total_beddays*100:.1f}% do total)")
    print(f"    LOS médio:    {sub['DIAS_PERM'].mean():.1f}d")
    print(f"    Mortalidade:  {sub['MORTE'].mean()*100:.2f}%")

# Administrative burden: D0 admissions that go through full admission process
d0_total = kidney[kidney["DIAS_PERM"] == 0]
print(f"\n  Carga administrativa (D0):")
print(f"    {len(d0_total):,} internações com alta no mesmo dia")
print(f"    Cada uma consome registro de admissão, leito, documentação e alta")
print(f"    Custo administrativo estimado: não capturado no VAL_TOT")

=== H9.3: IMPACTO — CUSTO vs LEITOS-DIA ===

  Alta suspeita (≥3):
    Internações:     6,614 (3.2% do total)
    Custo:        R$     937,190 (0.5% do total)
    Leitos-dia:          0 (0.0% do total)
    LOS médio:    0.0d
    Mortalidade:  0.09%

  Moderada (=2):
    Internações:    27,828 (13.5% do total)
    Custo:        R$   6,041,681 (3.2% do total)
    Leitos-dia:     62,382 (12.3% do total)
    LOS médio:    2.2d
    Mortalidade:  0.14%

  Carga administrativa (D0):
    26,023 internações com alta no mesmo dia
    Cada uma consome registro de admissão, leito, documentação e alta
    Custo administrativo estimado: não capturado no VAL_TOT


## 5. Cross-Reference: Eficiência vs Desnecessidade

In [8]:
# Load efficiency scores from nb05
try:
    eff_metrics = load_metrics("05_financial_analysis")
    print(f"Efficiency data available: {eff_metrics.get('hospitals_analyzed', 'N/A')} hospitals")
except Exception:
    print("No efficiency metrics found")

# Compute efficiency inline for cross-reference
proc_baselines = kidney.groupby("PROC_REA").agg(
    sys_n=("VAL_TOT", "count"),
    sys_median_cost=("VAL_TOT", "median"),
    sys_median_los=("DIAS_PERM", "median"),
).reset_index()
proc_baselines = proc_baselines[proc_baselines["sys_n"] >= 50]

hp = kidney[kidney["PROC_REA"].isin(proc_baselines["PROC_REA"])].copy()
hp = hp.merge(proc_baselines, on="PROC_REA", how="left")
hp["resolved"] = ((hp["MORTE"] == 0) & (hp["DIAS_PERM"] <= hp["sys_median_los"])).astype(int)

hp_agg = hp.groupby(["CNES", "PROC_REA"]).agg(
    n=("VAL_TOT", "count"),
    hosp_median_cost=("VAL_TOT", "median"),
    resolution_rate=("resolved", "mean"),
    sys_median_cost=("sys_median_cost", "first"),
).reset_index()
hp_agg = hp_agg[hp_agg["n"] >= 10]
hp_agg["cost_ratio"] = hp_agg["hosp_median_cost"] / hp_agg["sys_median_cost"]

hosp_eff = hp_agg.groupby("CNES").apply(
    lambda g: pd.Series({
        "eff_n": g["n"].sum(),
        "efficiency": np.average(g["resolution_rate"], weights=g["n"]) / 
                      np.average(g["cost_ratio"], weights=g["n"]),
        "w_resolution": np.average(g["resolution_rate"], weights=g["n"]),
    })
).reset_index()
hosp_eff = hosp_eff[hosp_eff["eff_n"] >= 30]

# Merge with unnecessary data
cross = hosp_unnec.merge(hosp_eff[["CNES", "efficiency", "w_resolution"]], on="CNES", how="inner")
print(f"\nHospitals with both efficiency and unnecessary data: {len(cross)}")

rho, p = stats.spearmanr(cross["efficiency"], cross["high_sus_pct"])
print(f"Efficiency vs % unnecessary: ρ = {rho:.3f}, p = {p:.2e}")
print(f"  → Hospitais eficientes {'têm MENOS' if rho < 0 else 'NÃO têm menos'} internações desnecessárias")

# 4-quadrant analysis
eff_med = cross["efficiency"].median()
unnec_med = cross["high_sus_pct"].median()
q1 = ((cross["efficiency"] >= eff_med) & (cross["high_sus_pct"] < unnec_med)).sum()
q2 = ((cross["efficiency"] >= eff_med) & (cross["high_sus_pct"] >= unnec_med)).sum()
q3 = ((cross["efficiency"] < eff_med) & (cross["high_sus_pct"] < unnec_med)).sum()
q4 = ((cross["efficiency"] < eff_med) & (cross["high_sus_pct"] >= unnec_med)).sum()
print(f"\nQuadrantes (eficiência × desnecessidade):")
print(f"  Eficiente + Poucas desnec.:    {q1} ({q1/len(cross)*100:.0f}%)")
print(f"  Eficiente + Muitas desnec.:    {q2} ({q2/len(cross)*100:.0f}%)")
print(f"  Ineficiente + Poucas desnec.:  {q3} ({q3/len(cross)*100:.0f}%)")
print(f"  Ineficiente + Muitas desnec.:  {q4} ({q4/len(cross)*100:.0f}%)")

Efficiency data available: 271 hospitals



Hospitals with both efficiency and unnecessary data: 271
Efficiency vs % unnecessary: ρ = 0.269, p = 7.14e-06
  → Hospitais eficientes NÃO têm menos internações desnecessárias

Quadrantes (eficiência × desnecessidade):
  Eficiente + Poucas desnec.:    55 (20%)
  Eficiente + Muitas desnec.:    81 (30%)
  Ineficiente + Poucas desnec.:  80 (30%)
  Ineficiente + Muitas desnec.:  55 (20%)


/var/folders/8r/2hn86n416n58v77nhrr2_mhw0000gn/T/ipykernel_80968/29390120.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hosp_eff = hp_agg.groupby("CNES").apply(


In [9]:
# --- Visualizations ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Efficiency vs unnecessary rate
axes[0].scatter(cross["efficiency"], cross["high_sus_pct"] * 100,
               s=cross["n"] / 15, alpha=0.5, color=COLORS["primary"])
axes[0].axvline(eff_med, color="gray", linestyle="--", linewidth=0.8)
axes[0].axhline(unnec_med * 100, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_xlabel("Score de Eficiência")
axes[0].set_ylabel("% Internações Suspeitas")
axes[0].set_title(f"Eficiência vs Desnecessidade (ρ={rho:.2f})")

# 2. Top 20 hospitals by unnecessary rate
top20 = hosp_unnec.head(20)
labels = [f"{n[:25]} ({t[:8]})" for n, t in zip(top20["name"], top20["legal_nature"])]
colors_bar = [COLORS["danger"] if p > 0.20 else COLORS["warning"] if p > 0.10 
              else COLORS["primary"] for p in top20["high_sus_pct"]]
axes[1].barh(range(len(labels)), top20["high_sus_pct"] * 100, color=colors_bar)
axes[1].set_yticks(range(len(labels)))
axes[1].set_yticklabels(labels, fontsize=7)
axes[1].set_xlabel("% Internações Suspeitas")
axes[1].set_title("Top 20 Hospitais")
axes[1].invert_yaxis()

# 3. Score distribution overall
score_counts = kidney["unnecessary_score"].value_counts().sort_index()
bar_colors = [COLORS["success"], COLORS["primary"], COLORS["warning"], 
              COLORS["danger"], COLORS["danger"], COLORS["danger"]]
axes[2].bar(score_counts.index, score_counts.values / 1000, 
           color=bar_colors[:len(score_counts)])
axes[2].set_xlabel("Score de Desnecessidade")
axes[2].set_ylabel("Internações (milhares)")
axes[2].set_title("Distribuição do Score")

fig.suptitle("Internações Potencialmente Desnecessárias — RQ9", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "unnecessary_overview", prefix="09")

  Saved: 13_unnecessary_overview.png


## 6. Análise Temporal — Internações Desnecessárias Estão Crescendo?

In [10]:
# Trend over time
yearly = kidney.groupby("year").agg(
    n=("CNES", "count"),
    high_sus_n=("unnecessary_score", lambda x: (x >= 3).sum()),
    high_sus_pct=("unnecessary_score", lambda x: (x >= 3).mean()),
    mean_score=("unnecessary_score", "mean"),
    pct_no_treatment=("flag_no_treatment", "mean"),
    pct_d0_nt=("flag_d0_no_therapy", "mean"),
    pct_therapeutic=("is_therapeutic", "mean"),
).reset_index()

print("=== TENDÊNCIA TEMPORAL ===")
print(f"{'Ano':>6s} {'N':>8s} {'Susp(n)':>8s} {'Susp%':>6s} {'Score':>6s} {'SemTer%':>7s} {'D0nt%':>6s} {'Terap%':>6s}")
print("-" * 65)
for _, r in yearly.iterrows():
    print(f"{r['year']:6.0f} {r['n']:>8,.0f} {r['high_sus_n']:>8,.0f} "
          f"{r['high_sus_pct']*100:5.1f}% {r['mean_score']:5.2f} "
          f"{r['pct_no_treatment']*100:6.1f}% {r['pct_d0_nt']*100:5.1f}% {r['pct_therapeutic']*100:5.1f}%")

# Kendall tau for trend
tau, p_tau = stats.kendalltau(yearly["year"], yearly["high_sus_pct"])
print(f"\nTendência (Kendall τ): τ = {tau:.3f}, p = {p_tau:.3f}")
if p_tau < 0.05:
    print(f"  → Tendência {'crescente' if tau > 0 else 'decrescente'} significativa")
else:
    print(f"  → Sem tendência significativa")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(yearly["year"], yearly["high_sus_pct"] * 100, 
            marker="o", color=COLORS["danger"], linewidth=2, label="% Suspeitas (≥3)")
axes[0].plot(yearly["year"], yearly["pct_no_treatment"] * 100, 
            marker="s", color=COLORS["warning"], linewidth=2, label="% Sem tratamento")
axes[0].plot(yearly["year"], yearly["pct_d0_nt"] * 100, 
            marker="^", color=COLORS["primary"], linewidth=2, label="% D0 não-terapêutico")
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("%")
axes[0].set_title("Evolução dos Indicadores")
axes[0].legend(fontsize=9)

axes[1].bar(yearly["year"], yearly["high_sus_n"], color=COLORS["danger"], alpha=0.7, label="Suspeitas")
axes[1].bar(yearly["year"], yearly["n"] - yearly["high_sus_n"], 
           bottom=yearly["high_sus_n"], color=COLORS["muted"], alpha=0.5, label="Demais")
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("Internações")
axes[1].set_title("Volume: Suspeitas vs Demais")
axes[1].legend(fontsize=9)

fig.suptitle("Tendência Temporal das Internações Desnecessárias", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "temporal_trend", prefix="09")

=== TENDÊNCIA TEMPORAL ===
   Ano        N  Susp(n)  Susp%  Score SemTer%  D0nt% Terap%
-----------------------------------------------------------------
  2015      640       12   1.9%  0.70   25.9%   2.2%  57.2%
  2016   14,234      447   3.1%  0.80   31.8%   4.1%  55.0%
  2017   15,002      460   3.1%  0.78   31.2%   3.9%  56.3%
  2018   16,362      590   3.6%  0.79   30.9%   4.7%  56.0%
  2019   17,757      554   3.1%  0.75   28.6%   4.2%  57.6%
  2020   16,664      602   3.6%  0.74   26.5%   5.2%  60.3%
  2021   17,144      646   3.8%  0.71   23.8%   6.2%  62.7%
  2022   20,588      680   3.3%  0.66   23.5%   5.9%  62.5%
  2023   25,762      776   3.0%  0.60   21.4%   5.4%  66.5%
  2024   30,985    1,017   3.3%  0.55   19.1%   6.4%  68.4%
  2025   31,362      830   2.6%  0.55   19.2%   6.2%  68.2%

Tendência (Kendall τ): τ = 0.055, p = 0.879
  → Sem tendência significativa
  Saved: 13_temporal_trend.png


## 7. Qual o Perfil do Paciente em Internações Desnecessárias?

In [11]:
# Patient demographics of high-suspicion admissions
high = kidney[kidney["unnecessary_score"] >= 3]
low = kidney[kidney["unnecessary_score"] == 0]

print("=== PERFIL DO PACIENTE: SCORE ≥ 3 vs SCORE = 0 ===")
metrics_list = [
    ("Idade média", lambda df: df["age"].mean(), ".1f"),
    ("% Masculino", lambda df: df["is_male"].mean() * 100, ".1f"),
    ("% Emergência", lambda df: df["is_emergency"].mean() * 100, ".1f"),
    ("% Migrado", lambda df: df["migrated"].mean() * 100, ".1f"),
    ("LOS médio", lambda df: df["DIAS_PERM"].mean(), ".1f"),
    ("Custo médio", lambda df: df["VAL_TOT"].mean(), ",.0f"),
    ("Mortalidade", lambda df: df["MORTE"].mean() * 100, ".3f"),
]

print(f"{'Métrica':25s} {'Suspeita (≥3)':>15s} {'Necessária (0)':>16s}")
print("-" * 60)
for label, fn, fmt in metrics_list:
    h_val = fn(high)
    l_val = fn(low)
    h_str = f"{h_val:{fmt}}"
    l_str = f"{l_val:{fmt}}"
    print(f"  {label:23s} {h_str:>15s} {l_str:>16s}")

# Age distribution comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(high["age"], bins=30, alpha=0.6, color=COLORS["danger"], 
            label=f"Suspeita (n={len(high):,})", density=True)
axes[0].hist(low["age"], bins=30, alpha=0.6, color=COLORS["success"], 
            label=f"Necessária (n={len(low):,})", density=True)
axes[0].set_xlabel("Idade")
axes[0].set_ylabel("Densidade")
axes[0].set_title("Distribuição Etária")
axes[0].legend(fontsize=9)

# Procedure mix comparison
cat_high = high["proc_category"].value_counts(normalize=True)
cat_low = low["proc_category"].value_counts(normalize=True)
cats = sorted(set(cat_high.index) | set(cat_low.index))
x = range(len(cats))
w = 0.35
axes[1].bar([i - w/2 for i in x], [cat_high.get(c, 0) * 100 for c in cats],
           w, color=COLORS["danger"], label="Suspeita", alpha=0.8)
axes[1].bar([i + w/2 for i in x], [cat_low.get(c, 0) * 100 for c in cats],
           w, color=COLORS["success"], label="Necessária", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(cats, rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("%")
axes[1].set_title("Mix de Procedimentos")
axes[1].legend(fontsize=9)

fig.suptitle("Perfil: Pacientes em Internações Suspeitas vs Necessárias", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "patient_profile", prefix="09")

=== PERFIL DO PACIENTE: SCORE ≥ 3 vs SCORE = 0 ===


Métrica                     Suspeita (≥3)   Necessária (0)
------------------------------------------------------------
  Idade média                        45.3             47.8
  % Masculino                        48.0             46.6
  % Emergência                       59.3             37.9
  % Migrado                          25.7             41.5
  LOS médio                           0.0              2.3
  Custo médio                         142            1,230
  Mortalidade                       0.091            0.254


  Saved: 13_patient_profile.png


## Summary Metrics

In [12]:
metrics = {
    "total_admissions": len(kidney),
    "high_suspicion_n": int((kidney["unnecessary_score"] >= 3).sum()),
    "high_suspicion_pct": round(float((kidney["unnecessary_score"] >= 3).mean() * 100), 1),
    "high_suspicion_cost": round(float(kidney[kidney["unnecessary_score"] >= 3]["VAL_TOT"].sum()), 0),
    "high_suspicion_cost_pct": round(float(kidney[kidney["unnecessary_score"] >= 3]["VAL_TOT"].sum() / total_cost * 100), 1),
    "d0_no_therapy": int(kidney["flag_d0_no_therapy"].sum()),
    "d0_total": int((kidney["DIAS_PERM"] == 0).sum()),
    "no_treatment_admissions": int(kidney["flag_no_treatment"].sum()),
    "hospitals_analyzed": len(hosp_unnec),
    "top_hospital_name": hosp_unnec.iloc[0]["name"] if len(hosp_unnec) > 0 else "",
    "top_hospital_pct": round(float(hosp_unnec.iloc[0]["high_sus_pct"] * 100), 1) if len(hosp_unnec) > 0 else 0,
    "efficiency_vs_unnecessary_rho": round(float(rho), 3),
    "efficiency_vs_unnecessary_p": float(p),
    "trend_tau": round(float(tau), 3),
    "trend_p": round(float(p_tau), 3),
}
save_metrics(metrics, "09_unnecessary_admissions")

print("\n=== SUMMARY ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

  Saved metrics: 13_unnecessary_admissions.json

=== SUMMARY ===
  total_admissions: 206500
  high_suspicion_n: 6614
  high_suspicion_pct: 3.2
  high_suspicion_cost: 937190.0
  high_suspicion_cost_pct: 0.5
  d0_no_therapy: 11159
  d0_total: 26023
  no_treatment_admissions: 50305
  hospitals_analyzed: 283
  top_hospital_name: HOSPITAL UNIVERSITARIO DA UFSCAR
  top_hospital_pct: 39.0
  efficiency_vs_unnecessary_rho: 0.269
  efficiency_vs_unnecessary_p: 7.14284699256401e-06
  trend_tau: 0.055
  trend_p: 0.879
